<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/OHDSI_Original25_AutoDiscovery_and_Reconstruction_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OHDSI Original 25-Patient Demo — Auto-Discovery & Reconstruction Helper

**Purpose:** Recover the exact files used for the submitted OHDSI software demonstration without requiring you to remember where they were stored.

This notebook searches Google Drive for:

- the **original 25-patient FHIR Bulk NDJSON cohort**
- the **OMOP SQLite database** whose clinical table counts match the submitted demo
- possible **Athena vocabulary folders/files**
- historical file paths embedded in the older project notebooks

It then ranks candidates against the submitted-demo signature:

| Item | Submitted reference |
|---|---:|
| FHIR Patient resources | 25 |
| OMOP `person` | 27 |
| `visit_occurrence` | 1,386 |
| `condition_occurrence` | 983 |
| `drug_exposure` | 1,275 |
| `observation` | 14,168 |
| `measurement` | 14,150 |
| `concept` | 4,066,375 |
| `concept_relationship` | 34,078,766 |

### Important safeguards

- It **does not overwrite** old files.
- It **does not regenerate a new Synthea cohort** and pretend it is the submitted one.
- It **does not select** the later analytical-stability databases with 1,071 persons / 25,000 clinical rows as the original demo.
- It only creates a small manifest pointing to recovered assets.
- If the original files cannot be found, it clearly says which evidence is missing.

## 0. Install small helper packages and clone/update the project repository

In [1]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "psutil"],
    check=True,
)

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

print("Repository:", REPO_DIR)

Repository: /content/ohdsi-fhir-omop-showcase-demo


## 1. Mount Google Drive

In [2]:
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; Drive mount skipped.")

MYDRIVE = Path("/content/drive/MyDrive")
print("MyDrive exists:", MYDRIVE.exists())

Mounted at /content/drive
MyDrive exists: True


## 2. Configuration and reference signature

You normally do **not** need to edit this cell.

Historical project notebooks previously used a Drive root named `fhir_omop_colab`, so that location is searched first. If nothing exact is found there, the notebook expands the search.

In [3]:
import os, json, sqlite3, re, gzip, time, math
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

REFERENCE_COUNTS = {
    "person": 27,
    "visit_occurrence": 1386,
    "condition_occurrence": 983,
    "drug_exposure": 1275,
    "observation": 14168,
    "measurement": 14150,
}

REFERENCE_VOCAB = {
    "vocabulary": 44,
    "concept": 4066375,
    "concept_relationship": 34078766,
    "concept_ancestor": 2163000,
    "concept_synonym": 2346129,
    "drug_strength": 3020774,
}

REFERENCE_FHIR_PATIENTS = 25

# Known later analytical-stability signature: never treat this as the submitted demo.
LATER_STABILITY_SIGNATURE = {
    "person": 1071,
    "visit_occurrence": 25000,
    "condition_occurrence": 25000,
    "drug_exposure": 25000,
    "measurement": 25000,
    "observation": 25000,
}

PREFERRED_ROOTS = [
    MYDRIVE / "fhir_omop_colab",
    MYDRIVE / "FHIR_OMOP_COLAB",
    MYDRIVE / "OHDSI_FHIR_OMOP",
]

RESULT_DIR = REPO_DIR / "results" / "original25_recovery"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Preferred roots:")
for p in PREFERRED_ROOTS:
    print(" -", p, "| exists:", p.exists())

Preferred roots:
 - /content/drive/MyDrive/fhir_omop_colab | exists: True
 - /content/drive/MyDrive/FHIR_OMOP_COLAB | exists: False
 - /content/drive/MyDrive/OHDSI_FHIR_OMOP | exists: False


## 3. Recover historical Drive path hints from the old notebooks

This does not trust notebook paths blindly. It simply extracts historical `/content/drive/MyDrive/...` strings and uses them as search hints.

In [4]:
path_pattern = re.compile(r"/content/drive/MyDrive/[^\"'\n\\]+")
hint_counter = Counter()

for nb_path in REPO_DIR.glob("*.ipynb"):
    try:
        raw = nb_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for match in path_pattern.findall(raw):
        clean = match.strip().rstrip("),.;")
        hint_counter[clean] += 1

hint_rows = [
    {"path_hint": p, "mentions": n}
    for p, n in hint_counter.most_common(100)
]
hints_df = pd.DataFrame(hint_rows)

print("Historical Drive path hints found:", len(hints_df))
if not hints_df.empty:
    display(hints_df.head(30))
    hints_df.to_csv(RESULT_DIR / "historical_drive_path_hints.csv", index=False)

Historical Drive path hints found: 54


,path_hint,mentions
0,/content/drive/MyDrive/MyDrive fhir_omop_colab,14
1,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,12
2,/content/drive/MyDrive/fhir_omop_colab/results...,9
3,/content/drive/MyDrive/MyDrive fhir_omop_colab...,8
4,/content/drive/MyDrive/,6
5,/content/drive/MyDrive/fhir_omop_colab/tfl_exe,6
6,/content/drive/MyDrive/fhir_omop_colab/results...,5
7,/content/drive/MyDrive/MCI_Project/mci-cardiom,5
8,/content/drive/MyDrive/GES_RAG_Temporal_Study/,5
9,/content/drive/MyDrive/MCI_Project/mci-cardiom...,5


# Part A — Recover the original OMOP SQLite database

## 4. Discover SQLite/database candidates

The search is performed in two stages:

1. historical project roots first
2. wider Google Drive search only if necessary

Only filename extensions commonly used for SQLite are collected.

In [5]:
DB_EXTENSIONS = {".sqlite", ".sqlite3", ".db"}

def discover_db_files(root, max_files=5000):
    found = []
    root = Path(root)
    if not root.exists():
        return found

    # Prune obviously irrelevant/high-volume folders.
    prune_names = {
        ".git", "node_modules", "__pycache__", ".venv", "venv",
        "site-packages", ".Trash", "Colab Notebooks"
    }

    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in prune_names]
        for fn in files:
            p = Path(current) / fn
            if p.suffix.lower() in DB_EXTENSIONS:
                try:
                    size = p.stat().st_size
                except Exception:
                    size = 0
                if size > 0:
                    found.append(p)
                    if len(found) >= max_files:
                        return found
    return found

preferred_db_candidates = []
for root in PREFERRED_ROOTS:
    preferred_db_candidates.extend(discover_db_files(root))

preferred_db_candidates = sorted(set(preferred_db_candidates))

print("Database candidates under preferred roots:", len(preferred_db_candidates))
for p in preferred_db_candidates[:50]:
    print(" -", p)

Database candidates under preferred roots: 14
 - /content/drive/MyDrive/fhir_omop_colab/results_25k/V0_clinical_core_25k.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/results_V1_V4_25k/V2_duplicate_encounter_ids_clinical_core_25k.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V0_tfl_encounter_corrected.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V1_tfl_encounter_corrected.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V2_tfl_encounter_corrected.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V3_tfl_encounter_corrected.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V4_tfl_encounter_corrected.sqlite
 - /content/drive/MyDrive/fhir_omop_colab/tfl_execution

## 5. Inspect clinical OMOP table counts and score each database

An **exact submitted-demo clinical signature** receives the highest score.

A database matching the later `1,071 / 25,000` analytical-stability signature is explicitly flagged and excluded from original-demo selection.

In [6]:
def sqlite_table_exists(conn, table):
    try:
        return conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=? LIMIT 1",
            (table,)
        ).fetchone() is not None
    except Exception:
        return False

def inspect_sqlite(path):
    row = {
        "path": str(path),
        "size_mb": round(path.stat().st_size / (1024**2), 3),
        "open_ok": False,
        "exact_submitted_clinical_signature": False,
        "later_stability_signature": False,
        "clinical_match_count": 0,
        "vocab_match_count": 0,
        "score": -999,
    }

    try:
        conn = sqlite3.connect(f"file:{path}?mode=ro", uri=True, timeout=20)
        row["open_ok"] = True

        clinical_matches = 0
        for table, expected in REFERENCE_COUNTS.items():
            if sqlite_table_exists(conn, table):
                observed = int(conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
                row[table] = observed
                if observed == expected:
                    clinical_matches += 1
            else:
                row[table] = None

        vocab_matches = 0
        for table, expected in REFERENCE_VOCAB.items():
            if sqlite_table_exists(conn, table):
                observed = int(conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
                row[table] = observed
                if observed == expected:
                    vocab_matches += 1
            else:
                row[table] = None

        row["clinical_match_count"] = clinical_matches
        row["vocab_match_count"] = vocab_matches
        row["exact_submitted_clinical_signature"] = clinical_matches == len(REFERENCE_COUNTS)

        stability_matches = 0
        for table, expected in LATER_STABILITY_SIGNATURE.items():
            if row.get(table) == expected:
                stability_matches += 1
        row["later_stability_signature"] = stability_matches >= 5

        # Ranking:
        # +100 per exact clinical table count
        # +20 per exact vocabulary table count
        # large bonus for all submitted clinical counts
        # large penalty for known later stability DBs
        score = clinical_matches * 100 + vocab_matches * 20
        if row["exact_submitted_clinical_signature"]:
            score += 1000
        if row["later_stability_signature"]:
            score -= 5000
        row["score"] = score

        conn.close()
    except Exception as e:
        row["error"] = repr(e)

    return row

def inspect_candidates(paths):
    rows = []
    for idx, p in enumerate(paths, 1):
        print(f"[{idx}/{len(paths)}] inspecting {p}")
        rows.append(inspect_sqlite(p))
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(
        ["score", "clinical_match_count", "vocab_match_count"],
        ascending=False
    ).reset_index(drop=True)

preferred_db_df = inspect_candidates(preferred_db_candidates)

if not preferred_db_df.empty:
    display(preferred_db_df.head(30))
    preferred_db_df.to_csv(RESULT_DIR / "preferred_root_database_candidates.csv", index=False)

exact_db_rows = (
    preferred_db_df[
        preferred_db_df["exact_submitted_clinical_signature"] == True
    ]
    if not preferred_db_df.empty else pd.DataFrame()
)

print("Exact submitted-demo DB matches under preferred roots:", len(exact_db_rows))

[1/14] inspecting /content/drive/MyDrive/fhir_omop_colab/results_25k/V0_clinical_core_25k.sqlite
[2/14] inspecting /content/drive/MyDrive/fhir_omop_colab/results_V1_V4_25k/V2_duplicate_encounter_ids_clinical_core_25k.sqlite
[3/14] inspecting /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V0_tfl_encounter_corrected.sqlite
[4/14] inspecting /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V1_tfl_encounter_corrected.sqlite
[5/14] inspecting /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V2_tfl_encounter_corrected.sqlite
[6/14] inspecting /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V3_tfl_encounter_corrected.sqlite
[7/14] inspecting /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/encounter_audit_repair_v10/fresh_omop_corrected/V4_tfl_encounter_corrected.sql

,path,size_mb,open_ok,exact_submitted_clinical_signature,later_stability_signature,clinical_match_count,vocab_match_count,score,person,visit_occurrence,condition_occurrence,drug_exposure,observation,measurement,vocabulary,concept,concept_relationship,concept_ancestor,concept_synonym,drug_strength
0,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
1,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
2,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
3,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.535,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
4,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
5,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.586,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
6,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.562,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
7,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.590,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
8,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.430,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0
9,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.434,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0


Exact submitted-demo DB matches under preferred roots: 0


## 6. Wider Drive search only if the exact database was not found

This may take a little longer on a large Drive. It does **not** read every file; it only inventories SQLite-like filenames and then inspects those candidates.

In [7]:
all_drive_db_df = pd.DataFrame()

if len(exact_db_rows) == 0:
    print("No exact submitted-demo DB found in preferred roots. Searching wider MyDrive...")
    all_db_candidates = discover_db_files(MYDRIVE)
    all_db_candidates = sorted(set(all_db_candidates))
    print("SQLite-like files found in MyDrive:", len(all_db_candidates))

    # Remove candidates already inspected.
    already = set(preferred_db_candidates)
    new_candidates = [p for p in all_db_candidates if p not in already]

    all_drive_db_df = inspect_candidates(new_candidates)
    if not all_drive_db_df.empty:
        display(all_drive_db_df.head(30))
        all_drive_db_df.to_csv(
            RESULT_DIR / "wider_drive_database_candidates.csv", index=False
        )

    combined_db_df = pd.concat(
        [preferred_db_df, all_drive_db_df],
        ignore_index=True
    ) if not preferred_db_df.empty or not all_drive_db_df.empty else pd.DataFrame()
else:
    combined_db_df = preferred_db_df.copy()

if not combined_db_df.empty:
    combined_db_df = combined_db_df.sort_values(
        ["score", "clinical_match_count", "vocab_match_count"],
        ascending=False
    ).reset_index(drop=True)
    combined_db_df.to_csv(RESULT_DIR / "all_database_candidates_ranked.csv", index=False)
    display(combined_db_df.head(20))

No exact submitted-demo DB found in preferred roots. Searching wider MyDrive...
SQLite-like files found in MyDrive: 14


,path,size_mb,open_ok,exact_submitted_clinical_signature,later_stability_signature,clinical_match_count,vocab_match_count,score,person,visit_occurrence,condition_occurrence,drug_exposure,observation,measurement,vocabulary,concept,concept_relationship,concept_ancestor,concept_synonym,drug_strength
0,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
1,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
2,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
3,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.535,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
4,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
5,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.586,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
6,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.562,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
7,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.590,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
8,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.430,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0
9,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.434,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0


## 7. Resolve the most likely original submitted-demo database

Selection is conservative:

- **AUTO-SELECT** only when all six submitted clinical table counts match exactly.
- If no exact match exists, report the best candidates but do **not** pretend one is the original.

In [8]:
RESOLVED_DB = None
DB_RESOLUTION_STATUS = "not_found"

if not combined_db_df.empty:
    exact = combined_db_df[
        (combined_db_df["exact_submitted_clinical_signature"] == True) &
        (combined_db_df["later_stability_signature"] == False)
    ]
    if len(exact) >= 1:
        RESOLVED_DB = Path(exact.iloc[0]["path"])
        DB_RESOLUTION_STATUS = "exact_submitted_clinical_signature"
    else:
        DB_RESOLUTION_STATUS = "no_exact_match"

print("DB resolution status:", DB_RESOLUTION_STATUS)
print("Resolved DB:", RESOLVED_DB)

if DB_RESOLUTION_STATUS == "no_exact_match":
    print("\nTop candidates are shown below for evidence only; none is auto-selected.")
    display(combined_db_df.head(10))

DB resolution status: no_exact_match
Resolved DB: None

Top candidates are shown below for evidence only; none is auto-selected.


,path,size_mb,open_ok,exact_submitted_clinical_signature,later_stability_signature,clinical_match_count,vocab_match_count,score,person,visit_occurrence,condition_occurrence,drug_exposure,observation,measurement,vocabulary,concept,concept_relationship,concept_ancestor,concept_synonym,drug_strength
0,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
1,/content/drive/MyDrive/fhir_omop_colab/results...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
2,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
3,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.535,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
4,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.531,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
5,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.586,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
6,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.562,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
7,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,27.590,True,False,True,0,0,-5000,1071,25000,25000,25000,25000,25000,0,0,0,0,0,0
8,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.430,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0
9,/content/drive/MyDrive/fhir_omop_colab/tfl_exe...,23.434,True,False,True,0,0,-5000,1071,0,25000,25000,25000,25000,0,0,0,0,0,0


# Part B — Recover the exact 25-patient FHIR Bulk folder

## 8. Discover likely Patient NDJSON files

To avoid reading every NDJSON file on Drive, filenames containing `patient` are searched first. Each candidate is then parsed to count actual FHIR `Patient` resources.

In [9]:
FHIR_EXTENSIONS = (".ndjson", ".jsonl", ".ndjson.gz", ".jsonl.gz")

def discover_patient_files(root, max_files=5000):
    found = []
    root = Path(root)
    if not root.exists():
        return found

    prune_names = {
        ".git", "node_modules", "__pycache__", ".venv", "venv",
        "site-packages", ".Trash"
    }

    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in prune_names]
        for fn in files:
            low = fn.lower()
            if "patient" in low and any(low.endswith(ext) for ext in FHIR_EXTENSIONS):
                found.append(Path(current) / fn)
                if len(found) >= max_files:
                    return found
    return found

def count_patient_resources(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    total_lines = 0
    patient_count = 0
    unique_ids = set()
    bad_json = 0

    try:
        with opener(path, "rt", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                total_lines += 1
                try:
                    obj = json.loads(line)
                except Exception:
                    bad_json += 1
                    continue

                if isinstance(obj, dict) and obj.get("resourceType") == "Patient":
                    patient_count += 1
                    if obj.get("id") is not None:
                        unique_ids.add(str(obj.get("id")))
                elif isinstance(obj, dict) and obj.get("resourceType") == "Bundle":
                    for entry in obj.get("entry", []) or []:
                        res = entry.get("resource", {})
                        if isinstance(res, dict) and res.get("resourceType") == "Patient":
                            patient_count += 1
                            if res.get("id") is not None:
                                unique_ids.add(str(res.get("id")))

        return {
            "path": str(path),
            "parent": str(path.parent),
            "size_mb": round(path.stat().st_size/(1024**2), 3),
            "nonempty_lines": total_lines,
            "patient_resources": patient_count,
            "unique_patient_ids": len(unique_ids),
            "bad_json_lines": bad_json,
            "exact_25_patient_match": patient_count == REFERENCE_FHIR_PATIENTS,
        }
    except Exception as e:
        return {
            "path": str(path),
            "parent": str(path.parent),
            "error": repr(e),
            "patient_resources": None,
            "unique_patient_ids": None,
            "exact_25_patient_match": False,
        }

preferred_patient_files = []
for root in PREFERRED_ROOTS:
    preferred_patient_files.extend(discover_patient_files(root))
preferred_patient_files = sorted(set(preferred_patient_files))

print("Patient-like NDJSON candidates under preferred roots:", len(preferred_patient_files))

patient_rows = []
for idx, p in enumerate(preferred_patient_files, 1):
    print(f"[{idx}/{len(preferred_patient_files)}] parsing {p}")
    patient_rows.append(count_patient_resources(p))

preferred_fhir_df = pd.DataFrame(patient_rows)
if not preferred_fhir_df.empty:
    preferred_fhir_df = preferred_fhir_df.sort_values(
        ["exact_25_patient_match", "patient_resources"],
        ascending=False
    )
    display(preferred_fhir_df.head(30))
    preferred_fhir_df.to_csv(
        RESULT_DIR / "preferred_root_patient_files.csv", index=False
    )

exact_patient_rows = (
    preferred_fhir_df[preferred_fhir_df["exact_25_patient_match"] == True]
    if not preferred_fhir_df.empty else pd.DataFrame()
)
print("Exact 25-patient file matches under preferred roots:", len(exact_patient_rows))

Patient-like NDJSON candidates under preferred roots: 0
Exact 25-patient file matches under preferred roots: 0


## 9. Wider Drive Patient-file search only if needed

In [10]:
all_drive_fhir_df = pd.DataFrame()

if len(exact_patient_rows) == 0:
    print("No exact 25-patient Patient file found in preferred roots. Searching wider MyDrive...")
    all_patient_files = sorted(set(discover_patient_files(MYDRIVE)))
    already = set(preferred_patient_files)
    new_patient_files = [p for p in all_patient_files if p not in already]

    rows = []
    for idx, p in enumerate(new_patient_files, 1):
        print(f"[{idx}/{len(new_patient_files)}] parsing {p}")
        rows.append(count_patient_resources(p))

    all_drive_fhir_df = pd.DataFrame(rows)
    if not all_drive_fhir_df.empty:
        all_drive_fhir_df = all_drive_fhir_df.sort_values(
            ["exact_25_patient_match", "patient_resources"],
            ascending=False
        )
        display(all_drive_fhir_df.head(30))
        all_drive_fhir_df.to_csv(
            RESULT_DIR / "wider_drive_patient_files.csv", index=False
        )

combined_fhir_df = pd.concat(
    [preferred_fhir_df, all_drive_fhir_df],
    ignore_index=True
) if not preferred_fhir_df.empty or not all_drive_fhir_df.empty else pd.DataFrame()

if not combined_fhir_df.empty:
    combined_fhir_df = combined_fhir_df.sort_values(
        ["exact_25_patient_match", "patient_resources"],
        ascending=False
    ).reset_index(drop=True)
    combined_fhir_df.to_csv(
        RESULT_DIR / "all_patient_files_ranked.csv", index=False
    )

No exact 25-patient Patient file found in preferred roots. Searching wider MyDrive...
[1/5] parsing /content/drive/MyDrive/MyDrive fhir_omop_colab /V1_missing_demographics_clinical_core_25k/Patient.ndjson
[2/5] parsing /content/drive/MyDrive/MyDrive fhir_omop_colab /V2_duplicate_encounter_ids_clinical_core_25k/Patient.ndjson
[3/5] parsing /content/drive/MyDrive/MyDrive fhir_omop_colab /V3_conflicting_codings_clinical_core_25k/Patient.ndjson
[4/5] parsing /content/drive/MyDrive/MyDrive fhir_omop_colab /V4_missing_medications_clinical_core_25k/Patient.ndjson
[5/5] parsing /content/drive/MyDrive/fhiry_pyomop_jamia_realdata/ohdsi-fhir-omop-showcase-demo/data_public_deidentified_fhir/mimic_iv_fhir_demo/MimicPatient.ndjson.gz


,path,parent,size_mb,nonempty_lines,patient_resources,unique_patient_ids,bad_json_lines,exact_25_patient_match
0,/content/drive/MyDrive/MyDrive fhir_omop_colab...,/content/drive/MyDrive/MyDrive fhir_omop_colab...,3.383,1071,1071,1071,0,False
1,/content/drive/MyDrive/MyDrive fhir_omop_colab...,/content/drive/MyDrive/MyDrive fhir_omop_colab...,3.388,1071,1071,1071,0,False
2,/content/drive/MyDrive/MyDrive fhir_omop_colab...,/content/drive/MyDrive/MyDrive fhir_omop_colab...,3.388,1071,1071,1071,0,False
3,/content/drive/MyDrive/MyDrive fhir_omop_colab...,/content/drive/MyDrive/MyDrive fhir_omop_colab...,3.388,1071,1071,1071,0,False
4,/content/drive/MyDrive/fhiry_pyomop_jamia_real...,/content/drive/MyDrive/fhiry_pyomop_jamia_real...,0.006,100,100,100,0,False


## 10. Resolve the likely original FHIR Bulk folder

If multiple folders contain exactly 25 Patient resources, the notebook does not silently assume they are identical. It inventories the other FHIR resource files in each matching parent folder so you can distinguish complete exports from copies/partial folders.

In [11]:
def inventory_fhir_folder(folder):
    folder = Path(folder)
    counts = []
    for p in sorted(folder.iterdir()) if folder.exists() else []:
        low = p.name.lower()
        if p.is_file() and any(low.endswith(ext) for ext in FHIR_EXTENSIONS):
            counts.append({
                "folder": str(folder),
                "file": p.name,
                "size_mb": round(p.stat().st_size/(1024**2), 3)
            })
    return counts

RESOLVED_FHIR_ROOT = None
FHIR_RESOLUTION_STATUS = "not_found"

if not combined_fhir_df.empty:
    exact25 = combined_fhir_df[
        combined_fhir_df["exact_25_patient_match"] == True
    ].copy()

    if len(exact25) == 1:
        RESOLVED_FHIR_ROOT = Path(exact25.iloc[0]["parent"])
        FHIR_RESOLUTION_STATUS = "single_exact_25_patient_folder"
    elif len(exact25) > 1:
        folder_inventory_rows = []
        for parent in sorted(exact25["parent"].unique()):
            folder_inventory_rows.extend(inventory_fhir_folder(parent))
        folder_inventory_df = pd.DataFrame(folder_inventory_rows)

        if not folder_inventory_df.empty:
            folder_summary = (
                folder_inventory_df.groupby("folder")
                .agg(
                    ndjson_files=("file", "count"),
                    total_mb=("size_mb", "sum")
                )
                .reset_index()
                .sort_values(["ndjson_files", "total_mb"], ascending=False)
            )
            display(folder_summary)
            folder_summary.to_csv(
                RESULT_DIR / "exact25_folder_inventory_summary.csv", index=False
            )

            # Conservative auto-selection:
            # choose a unique folder with the largest count of bulk resource files.
            max_files = folder_summary["ndjson_files"].max()
            leaders = folder_summary[folder_summary["ndjson_files"] == max_files]
            if len(leaders) == 1:
                RESOLVED_FHIR_ROOT = Path(leaders.iloc[0]["folder"])
                FHIR_RESOLUTION_STATUS = "exact_25_patient_folder_with_richest_bulk_export"
            else:
                FHIR_RESOLUTION_STATUS = "multiple_exact_25_patient_folders_need_review"
        else:
            FHIR_RESOLUTION_STATUS = "multiple_exact_25_patient_folders_need_review"

print("FHIR resolution status:", FHIR_RESOLUTION_STATUS)
print("Resolved FHIR root:", RESOLVED_FHIR_ROOT)

FHIR resolution status: not_found
Resolved FHIR root: None


# Part C — Check for Athena vocabulary assets

## 11. Locate candidate Athena vocabulary directories

This searches for directories containing common OHDSI vocabulary filenames such as `CONCEPT.csv` and `CONCEPT_RELATIONSHIP.csv`.

No vocabulary files are copied or uploaded.

In [12]:
ATHENA_REQUIRED_NAMES = {
    "concept.csv",
    "concept_relationship.csv",
    "vocabulary.csv",
}

def discover_athena_dirs(root, max_dirs=100):
    root = Path(root)
    if not root.exists():
        return []

    hits = []
    for current, dirs, files in os.walk(root):
        names = {f.lower() for f in files}
        overlap = ATHENA_REQUIRED_NAMES & names
        if len(overlap) >= 2:
            hits.append(Path(current))
            if len(hits) >= max_dirs:
                break
    return hits

athena_dirs = []
for root in PREFERRED_ROOTS:
    athena_dirs.extend(discover_athena_dirs(root))

athena_dirs = sorted(set(athena_dirs))

if not athena_dirs:
    print("No Athena folder found under preferred roots; performing wider MyDrive search...")
    athena_dirs = sorted(set(discover_athena_dirs(MYDRIVE)))

athena_rows = []
for d in athena_dirs:
    files = {p.name.lower(): p for p in d.iterdir() if p.is_file()}
    athena_rows.append({
        "path": str(d),
        "has_concept": "concept.csv" in files,
        "has_concept_relationship": "concept_relationship.csv" in files,
        "has_vocabulary": "vocabulary.csv" in files,
        "has_concept_ancestor": "concept_ancestor.csv" in files,
        "has_concept_synonym": "concept_synonym.csv" in files,
        "has_drug_strength": "drug_strength.csv" in files,
    })

athena_df = pd.DataFrame(athena_rows)
print("Athena-like directories found:", len(athena_df))
if not athena_df.empty:
    display(athena_df)
    athena_df.to_csv(RESULT_DIR / "athena_directory_candidates.csv", index=False)

No Athena folder found under preferred roots; performing wider MyDrive search...
Athena-like directories found: 0


# Part D — Verify the recovered pair

## 12. Compare recovered FHIR patients with OMOP PERSON provenance

If both an exact 25-patient FHIR folder and an exact submitted-demo OMOP database are found, this cell performs the first reviewer-critical reconciliation.

It checks whether `person_source_value` can be matched directly to FHIR `Patient.id`.

If direct provenance is unavailable, it reports that instead of guessing.

In [13]:
def iter_fhir_resources(folder):
    folder = Path(folder)
    for p in folder.iterdir():
        if not p.is_file():
            continue
        low = p.name.lower()
        if not any(low.endswith(ext) for ext in FHIR_EXTENSIONS):
            continue

        opener = gzip.open if low.endswith(".gz") else open
        try:
            with opener(p, "rt", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except Exception:
                        continue
                    if isinstance(obj, dict) and obj.get("resourceType"):
                        yield obj
        except Exception:
            pass

def norm_id(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    if s.startswith("urn:uuid:"):
        s = s[len("urn:uuid:"):]
    if "/" in s:
        parts = [x for x in s.split("/") if x]
        if len(parts) >= 2 and parts[-2].lower() == "patient":
            s = parts[-1]
    return s

reconciliation_df = pd.DataFrame()

if RESOLVED_DB and RESOLVED_FHIR_ROOT:
    fhir_ids = []
    for res in iter_fhir_resources(RESOLVED_FHIR_ROOT):
        if res.get("resourceType") == "Patient":
            fhir_ids.append(norm_id(res.get("id")))

    fhir_ids = [x for x in fhir_ids if x]
    fhir_set = set(fhir_ids)

    conn = sqlite3.connect(f"file:{RESOLVED_DB}?mode=ro", uri=True)
    cols = [
        r[1] for r in conn.execute('PRAGMA table_info("person")').fetchall()
    ]

    if "person_source_value" in cols:
        person = pd.read_sql_query(
            'SELECT person_id, person_source_value FROM person', conn
        )
        person["normalized_source_id"] = person["person_source_value"].map(norm_id)
        omop_set = set(person["normalized_source_id"].dropna().astype(str))

        reconciliation_df = pd.DataFrame([{
            "fhir_patient_resources": len(fhir_ids),
            "fhir_unique_patient_ids": len(fhir_set),
            "omop_person_rows": len(person),
            "omop_unique_nonnull_person_source_values": len(omop_set),
            "matched_unique_ids": len(fhir_set & omop_set),
            "fhir_only_unique_ids": len(fhir_set - omop_set),
            "omop_only_unique_ids": len(omop_set - fhir_set),
            "null_person_source_value_rows": int(person["normalized_source_id"].isna().sum()),
            "direct_reconciliation_available": True,
        }])

        # Keep exact IDs out of public output.
        private_dir = Path("/content/OHDSI_ORIGINAL25_PRIVATE")
        private_dir.mkdir(exist_ok=True)
        pd.DataFrame({"fhir_only_id": sorted(fhir_set - omop_set)}).to_csv(
            private_dir / "fhir_only_ids_PRIVATE.csv", index=False
        )
        pd.DataFrame({"omop_only_id": sorted(omop_set - fhir_set)}).to_csv(
            private_dir / "omop_only_ids_PRIVATE.csv", index=False
        )
    else:
        reconciliation_df = pd.DataFrame([{
            "fhir_patient_resources": len(fhir_ids),
            "fhir_unique_patient_ids": len(fhir_set),
            "omop_person_rows": int(conn.execute("SELECT COUNT(*) FROM person").fetchone()[0]),
            "direct_reconciliation_available": False,
            "reason": "person_source_value is absent; source-ID lineage cannot be directly tested"
        }])

    conn.close()
    display(reconciliation_df)
    reconciliation_df.to_csv(
        RESULT_DIR / "original25_person_reconciliation_summary.csv",
        index=False
    )
else:
    print("Exact FHIR+OMOP pair not yet resolved; reconciliation skipped.")

Exact FHIR+OMOP pair not yet resolved; reconciliation skipped.


## 13. Recompute submitted mapping percentages if the exact database is recovered

This is the second reviewer-critical check.

In [14]:
MAPPING = {
    "condition_occurrence": "condition_concept_id",
    "drug_exposure": "drug_concept_id",
    "measurement": "measurement_concept_id",
    "observation": "observation_concept_id",
    "visit_occurrence": "visit_concept_id",
}

mapping_rows = []

if RESOLVED_DB:
    conn = sqlite3.connect(f"file:{RESOLVED_DB}?mode=ro", uri=True)

    for table, concept_col in MAPPING.items():
        if not sqlite_table_exists(conn, table):
            continue
        cols = [r[1] for r in conn.execute(f'PRAGMA table_info("{table}")').fetchall()]
        if concept_col not in cols:
            continue

        total, mapped = conn.execute(
            f'''SELECT
                  COUNT(*),
                  SUM(CASE
                        WHEN "{concept_col}" IS NOT NULL AND "{concept_col}" <> 0
                        THEN 1 ELSE 0
                      END)
                FROM "{table}"'''
        ).fetchone()

        total = int(total or 0)
        mapped = int(mapped or 0)
        mapping_rows.append({
            "omop_table": table,
            "concept_column": concept_col,
            "total_records": total,
            "mapped_records": mapped,
            "unmapped_records": total - mapped,
            "mapped_percent": round(100 * mapped / total, 4) if total else np.nan,
        })

    conn.close()

mapping_df = pd.DataFrame(mapping_rows)
if not mapping_df.empty:
    display(mapping_df)
    mapping_df.to_csv(
        RESULT_DIR / "original25_mapping_completeness_recomputed.csv",
        index=False
    )
else:
    print("Mapping completeness could not be computed.")

Mapping completeness could not be computed.


# Part E — Create a standard manifest for the next reviewer-hardening notebook

## 14. Write `original25_recovery_manifest.json`

The manifest records where the recovered assets live. It does **not** copy the large database or restricted vocabulary files.

In [15]:
manifest = {
    "recovery_notebook": "OHDSI_Original25_AutoDiscovery_and_Reconstruction_COLAB.ipynb",
    "db_resolution_status": DB_RESOLUTION_STATUS,
    "resolved_omop_db": str(RESOLVED_DB) if RESOLVED_DB else None,
    "fhir_resolution_status": FHIR_RESOLUTION_STATUS,
    "resolved_fhir_root": str(RESOLVED_FHIR_ROOT) if RESOLVED_FHIR_ROOT else None,
    "athena_candidates": athena_df["path"].tolist() if not athena_df.empty else [],
    "submitted_reference": {
        "fhir_patient_resources": REFERENCE_FHIR_PATIENTS,
        "omop_clinical_counts": REFERENCE_COUNTS,
        "vocabulary_counts": REFERENCE_VOCAB,
    },
    "ready_for_reviewer_hardening": bool(RESOLVED_DB and RESOLVED_FHIR_ROOT),
}

manifest_path = RESULT_DIR / "original25_recovery_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))
print("\nSaved:", manifest_path)

{
  "recovery_notebook": "OHDSI_Original25_AutoDiscovery_and_Reconstruction_COLAB.ipynb",
  "db_resolution_status": "no_exact_match",
  "resolved_omop_db": null,
  "fhir_resolution_status": "not_found",
  "resolved_fhir_root": null,
  "athena_candidates": [],
  "submitted_reference": {
    "fhir_patient_resources": 25,
    "omop_clinical_counts": {
      "person": 27,
      "visit_occurrence": 1386,
      "condition_occurrence": 983,
      "drug_exposure": 1275,
      "observation": 14168,
      "measurement": 14150
    },
    "vocabulary_counts": {
      "vocabulary": 44,
      "concept": 4066375,
      "concept_relationship": 34078766,
      "concept_ancestor": 2163000,
      "concept_synonym": 2346129,
      "drug_strength": 3020774
    }
  },
  "ready_for_reviewer_hardening": false
}

Saved: /content/ohdsi-fhir-omop-showcase-demo/results/original25_recovery/original25_recovery_manifest.json


## 15. Final recovery status

Interpretation:

- **READY** → the exact clinical-count database and a 25-patient FHIR Bulk folder were recovered.
- **DB ONLY** → the submitted OMOP database was recovered, but the original FHIR export was not.
- **FHIR ONLY** → the 25-patient FHIR export was recovered, but the submitted OMOP database was not.
- **NOT RECOVERED** → neither exact asset was found; do not invent the 25→27 explanation.

In [16]:
if RESOLVED_DB and RESOLVED_FHIR_ROOT:
    final_status = "READY"
elif RESOLVED_DB:
    final_status = "DB ONLY"
elif RESOLVED_FHIR_ROOT:
    final_status = "FHIR ONLY"
else:
    final_status = "NOT RECOVERED"

summary = pd.DataFrame([{
    "status": final_status,
    "database": str(RESOLVED_DB) if RESOLVED_DB else None,
    "fhir_root": str(RESOLVED_FHIR_ROOT) if RESOLVED_FHIR_ROOT else None,
    "db_resolution": DB_RESOLUTION_STATUS,
    "fhir_resolution": FHIR_RESOLUTION_STATUS,
    "athena_candidate_count": len(athena_df),
    "next_action": (
        "Run reviewer hardening against recovered exact assets"
        if final_status == "READY"
        else "Review recovery evidence; do not claim a root cause yet"
    )
}])

display(summary)
summary.to_csv(RESULT_DIR / "original25_recovery_status.csv", index=False)

print("\n" + "="*72)
print("OHDSI ORIGINAL 25-PATIENT RECOVERY STATUS:", final_status)
print("="*72)

if final_status == "READY":
    print("Exact submitted-demo assets have been recovered.")
    print("Next: use the manifest with the reviewer-hardening notebook.")
elif final_status == "DB ONLY":
    print("Exact submitted-demo OMOP database recovered, but original FHIR export not yet found.")
elif final_status == "FHIR ONLY":
    print("Exact 25-patient FHIR export recovered, but exact submitted-demo OMOP database not yet found.")
else:
    print("Exact original assets were not recovered from Drive.")
    print("Do not regenerate a new random cohort and call it the submitted cohort.")

,status,database,fhir_root,db_resolution,fhir_resolution,athena_candidate_count,next_action
0,NOT RECOVERED,None,None,no_exact_match,not_found,0,Review recovery evidence; do not claim a root ...



OHDSI ORIGINAL 25-PATIENT RECOVERY STATUS: NOT RECOVERED
Exact original assets were not recovered from Drive.
Do not regenerate a new random cohort and call it the submitted cohort.


## 16. GitHub-safe files produced by this notebook

Under:

```text
results/original25_recovery/
```

the notebook may create:

- `historical_drive_path_hints.csv`
- `preferred_root_database_candidates.csv`
- `wider_drive_database_candidates.csv`
- `all_database_candidates_ranked.csv`
- `preferred_root_patient_files.csv`
- `wider_drive_patient_files.csv`
- `all_patient_files_ranked.csv`
- `exact25_folder_inventory_summary.csv`
- `athena_directory_candidates.csv`
- `original25_person_reconciliation_summary.csv`
- `original25_mapping_completeness_recomputed.csv`
- `original25_recovery_manifest.json`
- `original25_recovery_status.csv`

Exact row-level IDs are kept under `/content/OHDSI_ORIGINAL25_PRIVATE/` and should not be pushed automatically.

### After running

Push this notebook plus the GitHub-safe `results/original25_recovery/` outputs. Then the recovered manifest can drive the reviewer-hardening analysis without you manually entering file paths.